In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

In [2]:
class ConditionalAutoencoder(nn.Module):
    def __init__(self, input_dim, label_dim, latent_dim):
        super(ConditionalAutoencoder, self).__init__()
        self.input_dim = input_dim
        self.label_dim = label_dim
        self.latent_dim = latent_dim

        # Encoder
        self.fc1 = nn.Linear(input_dim + label_dim, 512)
        self.fc2 = nn.Linear(512, latent_dim)

        # Decoder
        self.fc3 = nn.Linear(latent_dim + label_dim, 512)
        self.fc4 = nn.Linear(512, input_dim)

    def encode(self, x, c):
        x = torch.cat([x, c], dim=1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

    def decode(self, z, c):
        z = torch.cat([z, c], dim=1)
        z = F.relu(self.fc3(z))
        return torch.sigmoid(self.fc4(z))

    def forward(self, x, c):
        z = self.encode(x, c)
        return self.decode(z, c)

In [3]:
# Load the MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=64, shuffle=True)

In [4]:
# Initialize the model and optimizer
model = ConditionalAutoencoder(input_dim=784, label_dim=10, latent_dim=20)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Loss function
criterion = nn.MSELoss()

In [ ]:
# Training loop
for epoch in range(10):  # number of epochs
    for data, labels in train_loader:
        # Flatten the data and one-hot encode the labels
        data = data.view(data.size(0), -1)
        labels = F.one_hot(labels, num_classes=10)

        # Zero the gradients
        optimizer.zero_grad()

        # Forward pass
        recon_data = model(data, labels.float())

        # Compute the loss
        loss = criterion(recon_data, data)

        # Backward pass
        loss.backward()

        # Update the weights
        optimizer.step()

    # Print the loss for this epoch
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


Epoch 1, Loss: 0.9309357404708862


In [ ]:
# Now that the model is trained, we can use it to generate reconstructions
# for arbitrary labels. For example, let's generate reconstructions for
# the first 10 data points, but with the label for "5" for all of them.

# Get the first 10 data points and their labels
data, _ = next(iter(train_loader))
data = data[:10].view(data.size(0), -1)

# Create labels for "5"
labels = torch.zeros(10, 10)
labels[:, 5] = 1

# Generate reconstructions
recon_data = model(data, labels.float())

# The variable `recon_data` now contains the reconstructions of the first
# 10 data points as if they all had the label "5".